# Qoro × PushQuantum Hackathon — 00 · Main notebook: setup and QUBO 101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/00_main_qubo_101.ipynb)

**Your mission:** choose one of two problems from the
[Quantum Optimization Benchmarking Library (QOBLIB)](https://github.com/ZIB-AOPT/QOBLIB), push it to the
largest instance you can, and get as close as you can to the best results published on QOBLIB, or beat
them. Your main tool is **partitioning**: cutting a problem that is too big for one quantum circuit into
many small programs that run in parallel on Qoro's cloud.

> **Step one: [sign up at dash.qoroquantum.net](https://dash.qoroquantum.net)** for your API key and
> hackathon credits. It takes a minute, and section 0.1 shows where the key goes.

| Notebook | What you'll do |
|---|---|
| **00 · Main** (this one) | **Part 0 · Setup:** API key, install, backend. **Part 1 · QUBO 101:** build a QUBO, *characterize and validate* it, solve it with QAOA and PCE, partition a bigger one. **Part 2:** choose your problem, and how to hand in |
| [**01 · Portfolio optimisation**](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/01_portfolio_optimisation.ipynb) (QOBLIB 06) | Official model → QUBO → classical baseline → PCE and partitioned QAOA → solution file |
| [**02 · Network design**](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/02_network_design.ipynb) (QOBLIB 08) | Topology QUBO + exact routing LP → sample, score, write a solution file |

Work through this notebook first, then pick **one** of 01 or 02. Setup is explained only here; the
problem notebooks repeat the setup code without the explanations.

**How the winner is chosen.** Two things count: the **scale** you reach and the **quality** of your
solution compared with the best results published on QOBLIB. Quality means QOBLIB's *official* objective,
not your QUBO's energy. [`qoblib_tools.py`](qoblib_tools.py) holds Python ports of QOBLIB's solution
checkers for both problems; they reproduce the published numbers exactly (checked against every
published reference solution). Treat it as the referee: change your models and solvers, not the referee.

### What is Divi?

[Divi](https://github.com/QoroQuantum/divi) is Qoro Quantum's open-source Python library for building and
running quantum programs at scale ([documentation](https://divi.readthedocs.io)). You describe the
problem, here a QUBO, and choose an algorithm such as QAOA or PCE. Divi generates the circuits, splits
large problems into many small programs, runs them in parallel on a local simulator or on Qoro's cloud,
and puts the results back together. In this hackathon you use it for exactly that: solving QUBOs that
are too big for one circuit by partitioning them.

---
## Part 0 · Setup

### 0.1 Get your API key

1. **Sign up at [dash.qoroquantum.net](https://dash.qoroquantum.net).** Your account comes with an API
   key and hackathon credits.
2. **Copy your API key** from the dashboard.
3. **In Google Colab, store it as a secret.** Click the key icon in Colab's left sidebar, add a secret
   named `QORO_API_KEY` with your key as its value, and switch on notebook access. Secrets belong to
   your Google account, so you add the key once; in each notebook you open later, switch on its access.

**Running locally instead?** Create a file called `.env` next to the notebooks containing
`QORO_API_KEY="your_api_key_here"`, and never commit it (add `.env` to your `.gitignore`). Divi finds it
on its own, and a key in a `.env` file wins over the other ways.

Without a key, everything still runs on the local simulator, and the cells that need the cloud print
"Skipped".

### 0.2 Install Divi

The cell below installs Divi **0.16.2**, the version these notebooks were tested with. It needs Python
3.11 or newer, which Colab has. Names can change between Divi's minor versions, so keep the pin.
**If the cell installs anything, restart the session** (Colab: *Runtime → Restart session*; Jupyter:
restart the kernel) and run the notebook again from the top. The second time, it finds Divi and skips
the install.

The `qubo-decompose` extra brings D-Wave's `hybrid` and `dwave-samplers` packages, used for QUBO
partitioning and simulated annealing. The `jupyter` extra lets Divi's progress displays render in the
notebook.

**Optional:** Maestro, Qoro's simulator, also works as a drop-in for code you write directly in Qiskit
(`pip install qiskit-maestro`) or PennyLane (`pip install pennylane-maestro`).

In [1]:
from importlib.metadata import PackageNotFoundError, version

try:
    divi_version = version("qoro-divi")
except PackageNotFoundError:
    divi_version = None

if divi_version == "0.16.2":
    print("qoro-divi 0.16.2 is installed.")
else:
    %pip install -q "qoro-divi[qubo-decompose,jupyter]==0.16.2"
    print("Installed Divi 0.16.2. Now restart the session (Runtime → Restart session) and run from the top.")

qoro-divi 0.16.1 is installed.


### 0.3 Load your key

In Colab, the cell below copies your secret into the environment, where Divi looks for it. Locally it
does nothing: Divi reads the `.env` file or the `QORO_API_KEY` environment variable by itself.

In [2]:
import os

try:  # Google Colab: copy the secret QORO_API_KEY into the environment
    from google.colab import userdata
    os.environ["QORO_API_KEY"] = userdata.get("QORO_API_KEY")
except ImportError:
    pass  # not in Colab: Divi reads .env or the environment by itself
except Exception as err:  # no secret yet, or notebook access switched off
    print(f"No QORO_API_KEY in Colab Secrets yet ({type(err).__name__}): see section 0.1.")

# Last resort, paste the key directly. Never share or commit a notebook with a real key in it.
# os.environ["QORO_API_KEY"] = "your_api_key_here"

### 0.4 Imports and backend

In [3]:
import math
import time
import warnings
from importlib.metadata import version

warnings.filterwarnings("ignore", category=SyntaxWarning)  # noisy docstrings in some dependencies

import dimod
import hybrid
import networkx as nx
import numpy as np
import requests
from dwave.samplers import SimulatedAnnealingSampler
from IPython.display import display

from divi.backends import JobConfig, MaestroSimulator, QoroService
from divi.backends.characterization import CharacterizationOptions, characterize_and_validate
from divi.qprog.algorithms import PCE, QAOA, DenseEncoding, PolyEncoding, SoftObjective
from divi.qprog.ensembles import BeamSearchStrategy, PartitioningProgramEnsemble
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog.problems import (
    BinaryOptimizationProblem,
    CommunityDecomposer,
    QUBOPartitioningConfig,
)

print("qoro-divi", version("qoro-divi"))

qoro-divi 0.16.1


Divi separates *what* you run from *where* it runs. Every quantum routine below takes a `backend`:

* `MaestroSimulator` — Qoro's simulator on your laptop. Great for small circuits and debugging.
* `QoroService` — Qoro's cloud (needs your key). Same code; partitioned problems run their sub-circuits in parallel.

The **Characterization Service** (section 1.2) always runs in the cloud, so it needs your key.

Every variational routine below tunes its angles with Divi's `MonteCarloOptimizer`. Each iteration
evaluates a whole population of `MC_POPULATION` = 50 parameter sets, that is 50 circuits sent to the
backend as **one batch**, then samples the next population around the best sets so far. Ten
iterations cost 500 circuits but only ten round trips to the cloud. A sequential optimiser such as
COBYLA sends one circuit per step and waits for each result, which is slow on a remote backend.
Raise `MC_POPULATION` or `MC_ITERATIONS` for harder problems; credits scale with the circuit count.

In [4]:
USE_CLOUD = False            # False: local Maestro simulator · True: Qoro cloud (uses credits)
RUN_CHARACTERIZATION = True  # Characterization Service calls (cloud; cost scales with QUBO size)
SHOTS = 10_000
MC_POPULATION = 50          # Monte Carlo optimiser: circuits per iteration, sent to the backend as one batch
MC_ITERATIONS = 10          # Monte Carlo optimiser iterations

try:
    service = QoroService()  # reads QORO_API_KEY from .env or the environment
    balance = service.get_credit_balance()
    print("Connected to Qoro. Credit balance:", balance.get("balance", balance))
except (ValueError, requests.RequestException) as err:  # no key, a wrong key, or no connection
    service = None
    print(f"{err}\nNo working API key (section 0.1): cloud cells will be skipped.")
if USE_CLOUD and service is None:
    raise RuntimeError("USE_CLOUD = True needs a working API key: see section 0.1.")


def make_backend(shots: int = SHOTS, force_sampling: bool = False):
    # force_sampling=True: estimate everything from shots instead of exact expectation values
    if USE_CLOUD:
        return QoroService(
            job_config=JobConfig(
                shots=shots, simulator_cluster="qoro_maestro", force_sampling=force_sampling
            )
        )
    return MaestroSimulator(shots=shots, force_sampling=force_sampling)


backend = make_backend()
print("Circuits run on:", type(backend).__name__)

Connected to Qoro. Credit balance: <your balance>
Circuits run on: MaestroSimulator


---
## Part 1 · QUBO 101 with Divi

A **QUBO** (quadratic unconstrained binary optimisation) problem asks for

$$\min_{x \in \{0,1\}^n} \; E(x) = \sum_i Q_{ii}\,x_i + \sum_{i<j} Q_{ij}\,x_i x_j .$$

Every problem in this hackathon ends up in this form, so let's learn the whole Divi toolchain on a
small one first.

### 1.1 Setting up the QUBO: number partitioning

Split numbers $a_1, \dots, a_n$ into two groups whose sums are as equal as possible. Let $x_i = 1$
mean "$a_i$ goes to group B". Then

$$E(x) = \Big(\sum_i a_i \;-\; 2\sum_i a_i x_i\Big)^2$$

is the squared difference of the two sums: 0 for a perfect split. Expanding the square (with
$x_i^2 = x_i$) gives linear and pairwise terms — a QUBO.

**This is the most important trick of the day:** squaring a linear expression turns any *linear
equality constraint* into a QUBO penalty that is zero exactly when the constraint holds. That's how
the capital/budget rules of the portfolio problem and the degree rules of the network problem get
into QUBO form later. `dimod`'s `add_linear_equality_constraint` does the expansion for you.

In [5]:
numbers = np.array([17, 5, 23, 11, 8, 31, 14, 2, 26, 9, 19, 6])
n_vars = len(numbers)

bqm = dimod.BinaryQuadraticModel(dimod.BINARY)
# Adds (Σ 2·a_i·x_i − Σ a_i)², expanded into linear + quadratic terms.
bqm.add_linear_equality_constraint(
    [(i, 2 * int(a)) for i, a in enumerate(numbers)],
    lagrange_multiplier=1.0,
    constant=-int(numbers.sum()),
)
print(f"{bqm.num_variables} variables, {bqm.num_interactions} couplings")
print("Q_00 =", bqm.get_linear(0), "  Q_01 =", bqm.get_quadratic(0, 1), "  offset =", bqm.offset)

# Quantum routines prefer coefficients of order 1. Rescaling keeps the same minimisers.
bqm_q = bqm.copy()
bqm_q.normalize()


def energy_of(model: dimod.BinaryQuadraticModel, bits) -> float:
    # Energy of a bitstring '0110…' (bit i = variable i) under a BQM.
    return model.energy({i: int(b) for i, b in enumerate(bits)})

12 variables, 66 couplings
Q_00 = -10472.0   Q_01 = 680.0   offset = 29241.0


With 12 variables there are only $2^{12} = 4096$ assignments, so we can brute-force the answer — a
luxury you won't have on the real problems, but perfect for checking a pipeline.

In [6]:
exact = dimod.ExactSolver().sample(bqm)
E_opt = exact.first.energy
optimal = [
    "".join(str(sample[i]) for i in range(n_vars))
    for sample, energy in exact.data(["sample", "energy"])
    if energy == E_opt
]
p_uniform = len(optimal) / 2**n_vars
print(f"Best energy {E_opt:.0f} → the two sums differ by {math.isqrt(int(E_opt))}")
print(f"{len(optimal)} optimal bitstrings out of {2**n_vars} (a random guess hits one with p = {p_uniform:.3f})")
print("Example:", optimal[0], "→", numbers[[b == "0" for b in optimal[0]]], "vs", numbers[[b == "1" for b in optimal[0]]])

Best energy 1 → the two sums differ by 1
94 optimal bitstrings out of 4096 (a random guess hits one with p = 0.023)
Example: 010111010110 → [17 23 14 26  6] vs [ 5 11  8 31  2  9 19]


### 1.2 Characterize & validate before you spend circuits

The **Qoro Characterization Service** analyses your QUBO in the cloud. One call to
`characterize_and_validate` compares QAOA against a classical baseline on *your* problem and returns
a report:

| Part of the report | What it tells you |
|---|---|
| `summary`, `recommendations` | One sentence on whether and how to run QAOA, and the advice behind it |
| `structure`, `fixed_values` | Variables that can be fixed exactly, independent parts, how well the QUBO splits into partitions |
| `classical` | The classical baseline's best solution, a lower bound, and the exact optimum when known |
| `qaoa`, `recommended_layers`, `gap_closed` | QAOA's quality per layer count, and how many layers to use |
| `angles`, `qaoa_initial_params()` | Optimised angles: warm-start QAOA and skip most of the optimiser loop |
| `references` | **Validation**: the energy of each reference state you sent and, when computed, its probability under QAOA |
| `penalty`, `recommended_lambda` | Penalty-weight guidance, when the problem has a penalty or constraints (notebook 01) |

Reference states are candidate solutions. They don't have to be proven optima.
`CharacterizationOptions` sets the QAOA ansatz you plan to run (`layers`, `mixer`).

Credits scale with the QUBO size, and the call below can take several minutes while the service analyses your QUBO. A finished report can be re-fetched for free with
`get_characterization_result(job_id, service=service)`.

In [7]:
np_problem = BinaryOptimizationProblem(bqm_q)  # same QUBO (and scale) that QAOA will see

report = None
if service is not None and RUN_CHARACTERIZATION:
    # Divi warns that the QUBO's constant offset isn't sent: reported energies exclude it.
    report = characterize_and_validate(
        np_problem,
        reference_states=optimal[:4],
        service=service,
        options=CharacterizationOptions(ansatz={"layers": 2}),
    )
    display(report)  # the full HTML report
    print(report)    # summary and recommendations as text
else:
    print("Skipped: no API key, or RUN_CHARACTERIZATION is False.")

/tmp/ipykernel_17181/4143755581.py:6: UserWarning: The problem carries a constant offset, which is not transmitted. The analysis is unaffected — a constant shifts every energy equally — but the reported energies exclude it.
  report = characterize_and_validate(


CharacterizationResult(job_id='3c42cd11-707e-48e1-94a6-128920bcb8b8', status='COMPLETED')
  The exact optimum is already known classically; use QAOA here to tune or benchmark it before scaling up.
  [action] The energy is unchanged by flipping every bit: fixing any one variable halves the search space and saves a qubit.
  [info] At the requested p = 2 layers QAOA's mean energy closes 85.9% of the gap from random guessing to the optimum.

CharacterizationResult(job_id='3c42cd11-707e-48e1-94a6-128920bcb8b8', status='COMPLETED')
  The exact optimum is already known classically; use QAOA here to tune or benchmark it before scaling up.
  [action] The energy is unchanged by flipping every bit: fixing any one variable halves the search space and saves a qubit.
  [info] At the requested p = 2 layers QAOA's mean energy closes 85.9% of the gap from random guessing to the optimum.


In [8]:
if report is not None:
    print("job id:             ", report.job_id)
    print("recommended layers: ", report.recommended_layers)
    print("gap closed at p=2:  ", report.gap_closed)  # 0 = random guessing, 1 = optimum
    print("classical baseline: ", report.classical)   # in the normalised QUBO's units
    print("fixed variables:    ", report.fixed_values)
    for ref in report.references:                      # validation of the reference states
        print("reference:          ", ref)

job id:              3c42cd11-707e-48e1-94a6-128920bcb8b8
recommended layers:  6
gap closed at p=2:   0.8588928793049482
classical baseline:  {'gap': 0.0, 'search': None, 'optimum': -1.6843317972350234, 'degeneracy': 94, 'best_energy': -1.6843317972350234, 'bound_label': 'exact enumeration', 'lower_bound': -1.6843317972350234, 'bound_method': 'exact', 'relative_gap': 0.0, 'best_feasible': None, 'random_energy': -1.4918202764976958, 'best_bitstring': '110111100000'}
fixed variables:     {}
reference:           {'energy': -1.6843317972350231, 'bitstring': '010111010110', 'gap_to_best': 2.220446049250313e-16, 'probability': 0.0006700985963810053}
reference:           {'energy': -1.6843317972350234, 'bitstring': '011101000101', 'gap_to_best': 0.0, 'probability': 0.0006734563708733472}
reference:           {'energy': -1.6843317972350236, 'bitstring': '110011000011', 'gap_to_best': -2.220446049250313e-16, 'probability': 0.0006740585242297733}
reference:           {'energy': -1.68433179723502

### 1.3 Solve it with QAOA

QAOA prepares a state by alternating the cost Hamiltonian (your QUBO) and a mixer, tunes the angles
with a classical optimiser, then samples bitstrings. Treat it as a **sampler**: judge it by the best
bitstrings it produces and how much probability it puts on good ones. `qaoa.solution` is only the
single most likely bitstring.

We run QAOA twice:
* **Blind**: Monte Carlo from random starting angles.
* **Warm-started**: from the service's angles at its recommended layer count, if you ran 1.2. The
  first population is those angles plus small random offsets.

In [9]:
def summarize(program, label: str) -> None:
    sols = program.get_top_solutions(n=2**n_vars)  # every sampled bitstring, most likely first
    energies = np.array([energy_of(bqm, s.bitstring) for s in sols])
    probs = np.array([s.prob for s in sols])
    p_opt = probs[energies == E_opt].sum()
    print(
        f"{label:<26} circuits={program.total_circuit_count:>4}  best sampled E={energies.min():>5.0f}  "
        f"P(optimal)={p_opt:.3f} ({p_opt / p_uniform:.1f}× random)  <E>={probs @ energies / probs.sum():>6.0f}"
    )


print(f"Random guessing: <E>={np.mean([energy_of(bqm, f'{k:012b}') for k in range(2**n_vars)]):.0f}\n")

qaoa = QAOA(
    np_problem,
    n_layers=2,
    optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
    max_iterations=MC_ITERATIONS,
    backend=backend,
    seed=1,
)
qaoa.run()
summarize(qaoa, "QAOA p=2, blind")

init = report.qaoa_initial_params() if report is not None else None
if init is not None:
    p = init.shape[1] // 2
    # One starting point per population member: the service's angles plus small random offsets.
    starts = init + np.random.default_rng(1).normal(scale=0.05, size=(MC_POPULATION, 2 * p))
    starts[0] = init[0]
    qaoa_ws = QAOA(
        np_problem,
        n_layers=p,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=backend,
        seed=1,
    )
    qaoa_ws.run(initial_params=starts)
    summarize(qaoa_ws, f"QAOA p={p}, warm-started")

Random guessing: <E>=3343



QAOA p=2, blind            circuits= 501  best sampled E=    1  P(optimal)=0.053 (2.3× random)  <E>=   741


QAOA p=6, warm-started     circuits= 501  best sampled E=    1  P(optimal)=0.088 (3.9× random)  <E>=   204


### 1.4 Solve it with PCE: more variables than qubits

**Pauli Correlation Encoding** stores each binary variable in the *sign of a multi-qubit Pauli
correlator* rather than in its own qubit. A variational circuit is trained so that the decoded
variables minimise your QUBO. That makes PCE a natural fit for QUBOs that are far too big for one
QAOA circuit.

You choose how variables map onto qubits (`encoding=`) and what the optimiser minimises (`objective=`):

| Encoding | Qubits for `N` variables |
|---|---|
| `DenseEncoding()` | $\lceil\log_2(N+1)\rceil$ |
| `PolyEncoding()` | about $\sqrt{2N}$ (each variable is a 1- or 2-qubit parity) |
| `MultiBasisEncoding(order=k)` | $q$ qubits hold $3\binom{q}{k}$ variables (X, Y and Z correlators) |

| Objective | What it does |
|---|---|
| `SoftObjective(alpha=…)` | Smooth relaxation; larger `alpha` pushes variables towards 0/1 |
| `CVaRObjective(fraction=…)` | Mean energy of the best sampled bitstrings; needs `make_backend(force_sampling=True)` |

`ProgressivePCE` raises `alpha` step by step until every variable is binary. Rank PCE's output by
energy (`sort_by="energy"`), not by probability.

In [10]:
for encoding in (DenseEncoding(), PolyEncoding()):
    pce = PCE(
        np_problem,
        encoding=encoding,
        objective=SoftObjective(alpha=4.0),
        n_layers=4,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=backend,
        seed=1,
    )
    pce.run()
    best = pce.get_top_solutions(n=10, min_prob=0.0, sort_by="energy")
    E_best = min(energy_of(bqm, s.bitstring) for s in best)
    print(
        f"PCE {type(encoding).__name__:<13}: {n_vars} variables on {pce.n_qubits} qubits, "
        f"{pce.total_circuit_count} circuits → best E = {E_best:.0f} (optimum {E_opt:.0f})"
    )

PCE DenseEncoding: 12 variables on 4 qubits, 501 circuits → best E = 25 (optimum 1)


PCE PolyEncoding : 12 variables on 5 qubits, 501 circuits → best E = 1 (optimum 1)


### 1.5 Too big for one circuit? Partition it

`PartitioningProgramEnsemble` is Divi's divide-and-conquer workflow:

1. **Decompose** the QUBO into sub-QUBOs. `CommunityDecomposer` finds communities in the QUBO's
   interaction graph, capped at `max_n_variables_per_cluster`. Any D-Wave `hybrid` decomposer, such as
   `hybrid.EnergyImpactDecomposer(size=…)`, also works.
2. **Solve** every sub-QUBO with its own QAOA or PCE program. They are independent, so on the cloud
   they run in parallel.
3. **Stitch** the per-partition candidates into global solutions with beam search.
4. **Polish**: `local_search=True` runs a greedy single-bit-flip descent on the *full* QUBO.

Let's try it on a 60-variable Max-Cut QUBO with community structure. Cutting edge $(u, v)$ means
$x_u \neq x_v$, so we minimise $-\sum_{(u,v)} w_{uv}\,(x_u + x_v - 2x_ux_v)$.

In [11]:
graph = nx.planted_partition_graph(4, 15, p_in=0.5, p_out=0.02, seed=7)
rng = np.random.default_rng(7)
maxcut = dimod.BinaryQuadraticModel(dimod.BINARY)
for u, v in graph.edges():
    w = rng.uniform(0.5, 1.5)
    maxcut.add_linear(u, -w)
    maxcut.add_linear(v, -w)
    maxcut.add_quadratic(u, v, 2 * w)
print(f"Max-Cut QUBO: {maxcut.num_variables} variables, {maxcut.num_interactions} couplings")

sa = SimulatedAnnealingSampler().sample(maxcut, num_reads=100, seed=1)
print(f"Simulated annealing (classical reference): E = {sa.first.energy:.3f}")


def solve_partitioned(bqm, routine="qaoa", cluster_size=12, n_layers=2, backend=backend, **routine_kwargs):
    # Partition `bqm`, solve every part with `routine`, and return the ensemble.
    # routine: "qaoa" | "iterative_qaoa" | "pce" | "progressive_pce"
    problem = BinaryOptimizationProblem(
        bqm,
        decomposer=CommunityDecomposer(
            QUBOPartitioningConfig(max_n_variables_per_cluster=cluster_size)
        ),
        composer=hybrid.SplatComposer(),
        local_search=True,
    )
    ensemble = PartitioningProgramEnsemble(
        problem=problem,
        quantum_routine=routine,
        n_layers=n_layers,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=backend,
        **routine_kwargs,
    )
    ensemble.run()
    return ensemble


partition_results = []
for routine, kwargs in [("qaoa", {}), ("pce", {"encoding": PolyEncoding(), "objective": SoftObjective(alpha=2.0)})]:
    t0 = time.time()
    ensemble = solve_partitioned(maxcut, routine, cluster_size=15, **kwargs)
    solution, energy = ensemble.aggregate_results(
        strategy=BeamSearchStrategy(beam_width=3, n_partition_candidates=5)
    )
    partition_results.append(
        f"Partitioned {routine.upper():<4}: {len(ensemble.programs)} sub-programs, "
        f"{ensemble.total_circuit_count} circuits, {time.time() - t0:.1f}s → E = {energy:.3f}"
    )
print("\n".join(partition_results))

Max-Cut QUBO: 60 variables, 219 couplings
Simulated annealing (classical reference): E = -161.949


Partitioned QAOA: 5 sub-programs, 2505 circuits, 6.0s → E = -158.606
Partitioned PCE : 5 sub-programs, 2505 circuits, 0.3s → E = -153.471


`ensemble.aggregate_results(...)` returns `(solution, energy)`, where `solution` follows
`bqm.variables` order. `ensemble.get_top_solutions(n, strategy=...)` returns several stitched
candidates — handy for post-processing.

**You now have the full toolchain.** Time for real benchmark problems: see the roadmap below.

---
## Part 2 · Choose your problem

Pick **one** problem and open its notebook. Each one starts with a short setup section (the same code
as Part 0, without the explanations) and assumes you have worked through Part 1.

| Notebook | Problem | How far it scales | Where partitioning comes in |
|---|---|---|---|
| [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/01_portfolio_optimisation.ipynb) `01_portfolio_optimisation.ipynb` | QOBLIB 06: multi-period portfolio optimisation | 3 to 400 assets, 2 to 15 days: 58 to about 36 000 binary variables. No published results above 50 assets | Partitioned QAOA on the full QUBO; PCE for whole QUBOs on few qubits |
| [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/02_network_design.ipynb) `02_network_design.ipynb` | QOBLIB 08: network design | `network05` to `network24`: 20 to 552 binary variables. Proven optima up to `network10` | The topology QUBO is partitioned; an exact LP scores each topology |

### Running a problem notebook in Colab

Each Colab notebook has its **own runtime**, so in the notebook you choose:

1. **Run the setup cells.** If the first one installs Divi, restart the session and run from the top,
   as in section 0.2. The setup also downloads the helper file `qoblib_tools.py`.
2. **Switch on notebook access** for your `QORO_API_KEY` secret (section 0.1).
3. **To keep your changes,** save a copy to your Drive (*File → Save a copy in Drive*).

Instance data is downloaded from QOBLIB's GitHub repository on first use. It, and the `.sol` files you
write, live in `/content` and disappear when the session ends: download what you want to keep.

---
## Handing in

**For the hackathon,** hand in each result the way the Qoro team announces at the event, with:

* the solution file (`.sol`) the problem notebook writes; our referee has already checked it;
* the instance, its size (binary variables, and qubits of your largest quantum program) and its
  official objective next to QOBLIB's best-known value;
* the method and settings that produced it, so it can be reproduced.

**Solution files** use QOBLIB's own formats. The problem notebooks write them for you; if you write your
own, follow QOBLIB's specification:

| Problem | Format | File name |
|---|---|---|
| Portfolio (01) | [Canonical format](https://github.com/ZIB-AOPT/QOBLIB/blob/main/06-portfolio/check/README.md#canonical-solution-format): header lines `instance`, `budget`, `lambda` (and optionally `objective`), then one `period symbol long short` line per position held | `<instance>_b<budget>_l<lambda>.sol`, for example `po_a010_t10_orig_b004_l1e-4.sol` |
| Network design (02) | [Gurobi-style format](https://github.com/ZIB-AOPT/QOBLIB/blob/main/08-network/check/README.md#input-format): `z`, then `x#i#j` for the links and `f#k#i#j` for the flows, 1-indexed, flows in units of 1/1000 | `network<NN>.sol`, for example `network12.sol` |

Keep the `instance`, `budget` and `lambda` lines of a portfolio file and the name of a network file: they
say which instance you solved. The referee recomputes the objective from your positions or flows, so the
value written in the file doesn't count.

**Submitting to QOBLIB itself (optional).** If you match or beat a best-known value, you can also submit
your result to the library; QOBLIB's
[contribution guide](https://github.com/ZIB-AOPT/QOBLIB/blob/main/CONTRIBUTING.md) explains how. Before
you do, run QOBLIB's own checker on your file (`cargo build --release` in the problem's `check/` folder).
Our referee matches it, but theirs is the official one.

Good luck, and have fun!